In [9]:
import os
import hashlib
from pathlib import Path
from PIL import Image
import imagehash
import pandas as pd

In [11]:
def phash_file(path, hash_size=16):
    """Compute a perceptual hash (phash) for an image."""
    try:
        img = Image.open(path)
        return imagehash.phash(img, hash_size=hash_size)
    except Exception as e:
        print(f"Error hashing {path}: {e}")
        return None

def collect_phashes(folder):
    """Walk folder and map phash → [file paths]."""
    folder = Path(folder)
    phashes = {}
    for fn in folder.rglob("*"):
        if fn.is_file():
            h = phash_file(fn)
            if h is not None:
                phashes.setdefault(str(h), []).append(str(fn))
    return phashes

# ── Configure your two image folders here ─────────────────────────
folder1 = "blueblobs_val"
folder2 = "blue_blob_train_images_new"

# ── Build phash maps ───────────────────────────────────────────────
ph1 = collect_phashes(folder1)
ph2 = collect_phashes(folder2)

# ── Find exact phash‐matches ───────────────────────────────────────
rows = []
for h in set(ph1) & set(ph2):
    for f1 in ph1[h]:
        for f2 in ph2[h]:
            rows.append((h, f1, f2))

if rows:
    df = pd.DataFrame(rows, columns=["phash", "file1", "file2"])
    display(df)
else:
    print("No exact perceptual‐hash duplicates found.")

# ── (Optional) find *near*-duplicates by Hamming distance threshold ──
THRESH = 5
near = []
for h1, files1 in ph1.items():
    for h2, files2 in ph2.items():
        d = imagehash.hex_to_hash(h1) - imagehash.hex_to_hash(h2)
        if 0 < d <= THRESH:
            for f1 in files1:
                for f2 in files2:
                    near.append((f1, f2, d))

if near:
    df2 = pd.DataFrame(near, columns=["file1", "file2", "hamming_dist"])
    print(f"\nNear-duplicates (d ≤ {THRESH}):")
    display(df2)

No exact perceptual‐hash duplicates found.
